# EDA — Dados do Inversor (painels.xlsx)
**Projeto Final — CKP8277 Aprendizagem Automática · UFC**

**Aluno:** Diego Melo — 603127

Dataset do inversor fotovoltaico: tensões/correntes MPPT DC, tensões/correntes/frequências CA, correntes por string, temperatura interna e potência gerada.
Objetivo: EDA completa orientada à regressão de potência (`Potência(W)`).

---
## 1. Configuração Global

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.feature_selection import mutual_info_regression

# ── Tema global ──────────────────────────────────────────────────────────────
PALETTE  = 'Set2'
PRIMARY  = '#2D7DD2'
ACCENT   = '#F18F01'
NEUTRAL  = '#6C757D'
BG       = '#F8F9FA'
GRID_CLR = '#DEE2E6'

sns.set_theme(
    style='whitegrid',
    palette=PALETTE,
    font='DejaVu Sans',
    rc={
        'figure.facecolor'  : BG,
        'axes.facecolor'    : BG,
        'axes.edgecolor'    : GRID_CLR,
        'axes.titlesize'    : 13,
        'axes.titleweight'  : 'bold',
        'axes.labelsize'    : 11,
        'xtick.labelsize'   : 9,
        'ytick.labelsize'   : 9,
        'legend.fontsize'   : 9,
        'figure.titlesize'  : 15,
        'figure.titleweight': 'bold',
    }
)

DPI   = 120
FIG_W = 15
FIG_H = 4.5

print('Seaborn', sns.__version__, '| Pandas', pd.__version__)

---
## 2. Carregamento e Inspeção Inicial

In [ ]:
DATA_PATH = 'data/painels.xlsx'

df_raw = pd.read_excel(DATA_PATH)
df_raw = df_raw.sort_values('hora').reset_index(drop=True)

# Grupos de features
MPPT_V  = ['V MPPT 1(V)', 'V MPPT 2(V)', 'V MPPT 3(V)', 'V MPPT 4(V)']
MPPT_I  = ['I MPPT 1(A)', 'I MPPT 2(A)', 'I MPPT 3(A)', 'I MPPT 4(A)']
AC_V    = ['Ua(V)', 'Ub(V)', 'Uc(V)']
AC_I    = ['I CA 1(A)', 'I CA 2(A)', 'I CA 3(A)']
AC_F    = ['F CA 1(Hz)', 'F CA 2(Hz)', 'F CA 3(Hz)']
STR_I   = [f'Istr{i}(A)' for i in range(1, 11)]
MISC    = ['Temperatura(℃)', 'RSSI(%)']
TARGET  = 'Potência(W)'
ALL_NUM = MPPT_V + MPPT_I + AC_V + AC_I + AC_F + STR_I + MISC

# Colunas temporais
df_raw['HOUR']  = df_raw['hora'].dt.hour
df_raw['MONTH'] = df_raw['hora'].dt.month
MONTH_MAP = {12:'Dez',1:'Jan',2:'Fev',3:'Mar',4:'Abr',5:'Mai',6:'Jun',7:'Jul',8:'Ago',9:'Set',10:'Out'}
df_raw['MONTH_NAME'] = df_raw['MONTH'].map(MONTH_MAP)
MONTH_ORDER = ['Dez','Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out']

print(f'Shape: {df_raw.shape}')
print(f'Período: {df_raw.hora.min()} → {df_raw.hora.max()}')
print()
print('Modo de operação:')
print(df_raw['Modo de operação'].value_counts().to_string())

In [ ]:
# Estatísticas descritivas
df_raw[[TARGET, 'Temperatura(℃)', 'RSSI(%)'] + MPPT_V[:2] + MPPT_I[:2]].describe().round(2)

In [ ]:
# Filtro: apenas modo Normal e potência > 0 para análise diurna
df = df_raw[df_raw['Modo de operação'] == 'Normal'].copy()
df_day = df[df[TARGET] > 0].copy()

print(f'Total registros          : {len(df_raw):,}')
print(f'Modo Normal              : {len(df):,}')
print(f'Normal + Potência > 0    : {len(df_day):,}')

---
## 3. Distribuição do Target e das Features Principais — Histograma + KDE

In [ ]:
KEY_FEATS = [TARGET, 'Temperatura(℃)'] + MPPT_V[:2] + MPPT_I[:2] + AC_I[:2]
COLORS = sns.color_palette(PALETTE, len(KEY_FEATS))

ncols = 4
nrows = int(np.ceil(len(KEY_FEATS) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(FIG_W, nrows * FIG_H * 0.85), dpi=DPI)
axes = axes.flatten()
fig.suptitle('Distribuição das Features-Chave e Target (modo Normal, Potência>0)')

for i, col in enumerate(KEY_FEATS):
    ax = axes[i]
    sns.histplot(data=df_day, x=col, kde=True, color=COLORS[i],
                 bins=40, edgecolor='white', linewidth=0.3, ax=ax)
    skew = df_day[col].skew()
    ax.set_title(col, pad=5)
    ax.set_xlabel('')
    ax.annotate(f'skew={skew:.2f}', xy=(0.97, 0.95), xycoords='axes fraction',
                ha='right', va='top', fontsize=8, color='dimgray')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

---
## 4. Distribuição por Modo de Operação — Boxplot por Classe

In [ ]:
# Potência + temperatura por modo de operação
fig, axes = plt.subplots(1, 2, figsize=(10, FIG_H), dpi=DPI)
fig.suptitle('Distribuição por Modo de Operação')

for ax, col in zip(axes, [TARGET, 'Temperatura(℃)']):
    sns.boxplot(data=df_raw, x='Modo de operação', y=col,
                order=['Normal','Parado','Fault'],
                palette='Set2', linewidth=0.8, fliersize=2,
                flierprops=dict(marker='o', alpha=0.3), ax=ax)
    ax.set_title(col); ax.set_xlabel('Modo')

plt.tight_layout()
plt.show()

In [ ]:
# Potência por mês
months_present = [m for m in MONTH_ORDER if m in df_day['MONTH_NAME'].unique()]

fig, ax = plt.subplots(figsize=(12, FIG_H), dpi=DPI)
sns.boxplot(data=df_day, x='MONTH_NAME', y=TARGET,
            order=months_present, palette=PALETTE,
            linewidth=0.8, fliersize=1.5,
            flierprops=dict(marker='o', alpha=0.2), ax=ax)
ax.set_title(f'{TARGET} por Mês')
ax.set_xlabel('Mês'); ax.set_ylabel(TARGET)
plt.tight_layout()
plt.show()

---
## 5. Violin Plot por Hora — Detecção de Outliers

In [ ]:
df_hr = df_day[(df_day['HOUR'] >= 5) & (df_day['HOUR'] <= 18)].copy()
VIOLIN_COLS = [TARGET, 'Temperatura(℃)'] + MPPT_I[:2] + AC_I[:1]

fig, axes = plt.subplots(len(VIOLIN_COLS), 1,
                         figsize=(FIG_W, len(VIOLIN_COLS) * 3.2), dpi=DPI)
fig.suptitle('Violin Plot por Hora — Distribuição e Outliers', y=1.005)

for ax, col in zip(axes, VIOLIN_COLS):
    sns.violinplot(data=df_hr, x='HOUR', y=col,
                   palette='Set2', inner='quartile',
                   linewidth=0.7, scale='width', ax=ax)
    q1, q3 = df_hr[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    n_out = ((df_hr[col] < q1 - 1.5*iqr) | (df_hr[col] > q3 + 1.5*iqr)).sum()
    ax.set_title(f'{col}  (outliers IQR: {n_out:,})', pad=5)
    ax.set_xlabel('Hora'); ax.set_ylabel(col)

plt.tight_layout()
plt.show()

---
## 6. Pair Plot — Grupos de Features vs Target

In [ ]:
# Amostra para não sobrecarregar
df_s = df_day.sample(n=min(2500, len(df_day)), random_state=42)

PAIR_COLS = MPPT_I[:3] + AC_I[:2] + ['Temperatura(℃)', TARGET]

g = sns.PairGrid(
    df_s[PAIR_COLS + ['MONTH_NAME']],
    vars=PAIR_COLS,
    hue='MONTH_NAME',
    hue_order=[m for m in MONTH_ORDER if m in df_s['MONTH_NAME'].unique()],
    palette=PALETTE,
    diag_sharey=False,
    height=2.0, aspect=1.0
)

g.map_diag(sns.kdeplot, fill=True, alpha=0.35, linewidth=1.1)

g.map_lower(sns.regplot,
    scatter_kws=dict(alpha=0.12, s=7, linewidths=0),
    line_kws=dict(linewidth=1.5),
    lowess=True, ci=None, color=PRIMARY
)

def pearson_text(x, y, **kwargs):
    ax = plt.gca()
    xy = pd.DataFrame({'x': x, 'y': y}).dropna()
    if len(xy) < 2 or xy['x'].std() == 0 or xy['y'].std() == 0:
        return
    r, p = stats.pearsonr(xy['x'], xy['y'])
    star = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))
    ax.cla()
    ax.set_axis_off()
    ax.text(0.5, 0.5, f'r={r:.2f}{star}',
            transform=ax.transAxes, ha='center', va='center',
            fontsize=10, color=PRIMARY if abs(r) > 0.5 else NEUTRAL, fontweight='bold')

g.map_upper(pearson_text)

g.add_legend(title='Mês', bbox_to_anchor=(1.02, 0.5), loc='center left')
g.figure.suptitle('Pair Plot — Correntes MPPT/CA, Temperatura vs Potência', y=1.01,
                   fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. Scatter Features vs Target + Linha de Tendência (Lowess)

In [ ]:
SCATTER_FEATS = MPPT_I[:4] + AC_I[:3] + ['Temperatura(℃)']
COLORS_S = sns.color_palette(PALETTE, len(SCATTER_FEATS))

ncols = 4
nrows = int(np.ceil(len(SCATTER_FEATS) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(FIG_W, nrows * FIG_H), dpi=DPI)
axes = axes.flatten()
fig.suptitle(f'Features vs {TARGET} — Scatter + Tendência (lowess)')

for i, col in enumerate(SCATTER_FEATS):
    ax = axes[i]
    sns.regplot(
        data=df_s, x=col, y=TARGET,
        scatter_kws=dict(alpha=0.18, s=8, color=COLORS_S[i], linewidths=0),
        line_kws=dict(color=ACCENT, linewidth=2),
        lowess=True, ci=None, ax=ax
    )
    r, _ = stats.pearsonr(df_day[col].dropna(), df_day[TARGET].dropna())
    ax.set_title(f'{col}\nr = {r:.3f}', pad=5)
    ax.set_xlabel(col)
    ax.set_ylabel(TARGET if i % ncols == 0 else '')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

---
## 8. Mapa de Calor de Correlação — Todos os Grupos

In [ ]:
# Remove Istr9 (constante / NaN)
valid_num = [c for c in ALL_NUM if df_day[c].std() > 0]
corr = df_day[valid_num + [TARGET]].corr()

fig, ax = plt.subplots(figsize=(16, 13), dpi=DPI)
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, vmin=-1, vmax=1,
    linewidths=0.3, linecolor='white',
    annot_kws={'size': 7},
    square=True, ax=ax
)
ax.set_title('Matriz de Correlação de Pearson — Todos os Grupos (modo Normal, Potência>0)')
plt.tight_layout()
plt.show()

---
## 9. Análise de Outliers — IQR + Z-score

In [ ]:
KEY_OUT = [TARGET, 'Temperatura(℃)'] + MPPT_I[:2] + AC_I[:2] + STR_I[:4]

outlier_rows = []
for col in KEY_OUT:
    q1, q3 = df_day[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    iqr_n = ((df_day[col] < q1 - 1.5*iqr) | (df_day[col] > q3 + 1.5*iqr)).sum()
    z_n   = (np.abs(stats.zscore(df_day[col].dropna())) > 3).sum()
    outlier_rows.append({'Feature': col, 'IQR outliers': iqr_n,
                          'IQR %': f'{100*iqr_n/len(df_day):.2f}%', 'Z>3 outliers': z_n})

print(pd.DataFrame(outlier_rows).to_string(index=False))

In [ ]:
# Boxplot Z-score comparativo
df_z = df_day[KEY_OUT].apply(lambda s: (s - s.mean()) / s.std()).melt(var_name='Feature', value_name='Z-score')

fig, ax = plt.subplots(figsize=(13, 4), dpi=DPI)
sns.boxplot(data=df_z, x='Feature', y='Z-score',
            palette=PALETTE, linewidth=0.7,
            flierprops=dict(marker='o', alpha=0.15, markersize=2), ax=ax)
ax.axhline(3,  color='red', linestyle='--', linewidth=1, label='|Z|=3')
ax.axhline(-3, color='red', linestyle='--', linewidth=1)
ax.set_title('Outliers Padronizados (Z-score) — Features-Chave')
ax.tick_params(axis='x', rotation=25)
ax.legend()
plt.tight_layout()
plt.show()

---
## 10. Seleção de Features — Pearson, Spearman e Informação Mútua

In [ ]:
feat_sel = [c for c in valid_num if c != TARGET]
X = df_day[feat_sel].values
y = df_day[TARGET].values

pearson_s  = df_day[feat_sel].corrwith(df_day[TARGET]).abs()
spearman_s = df_day[feat_sel].corrwith(df_day[TARGET], method='spearman').abs()
mi_raw     = mutual_info_regression(X, y, random_state=42)
# Normaliza MI para [0,1]
mi_norm    = mi_raw / (mi_raw.max() + 1e-9)
mi_s       = pd.Series(mi_norm, index=feat_sel)

score_df = pd.DataFrame({
    'Pearson |r|' : pearson_s,
    'Spearman |ρ|': spearman_s,
    'MI (norm)'   : mi_s
}).sort_values('Pearson |r|', ascending=False)

print('Ranking de Features (top 15 por Pearson):')
print(score_df.head(15).round(4).to_string())

In [ ]:
top15 = score_df.head(15).reset_index().melt(id_vars='index', var_name='Métrica', value_name='Score')
top15.columns = ['Feature', 'Métrica', 'Score']

fig, ax = plt.subplots(figsize=(13, 4.5), dpi=DPI)
sns.barplot(data=top15, x='Feature', y='Score',
            hue='Métrica', palette='Set2', ax=ax)
ax.set_title('Top 15 Features — Relevância em Relação a Potência(W)')
ax.tick_params(axis='x', rotation=30)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

---
## 11. Análise por Grupo de Features

In [ ]:
# Tensões MPPT (DC) por hora
hourly = df_day.groupby('HOUR')[MPPT_V + MPPT_I + [TARGET]].median().reset_index()

fig, axes = plt.subplots(1, 3, figsize=(FIG_W, FIG_H), dpi=DPI)
fig.suptitle('Perfil Diário Mediano — Tensões MPPT, Correntes MPPT e Potência')

pal4 = sns.color_palette(PALETTE, 4)

ax = axes[0]
for col, c in zip(MPPT_V, pal4):
    ax.plot(hourly['HOUR'], hourly[col], label=col, color=c, linewidth=1.8, marker='o', markersize=3)
ax.set_title('Tensões MPPT DC (V)'); ax.set_xlabel('Hora'); ax.legend(fontsize=8)

ax = axes[1]
for col, c in zip(MPPT_I, pal4):
    ax.plot(hourly['HOUR'], hourly[col], label=col, color=c, linewidth=1.8, marker='o', markersize=3)
ax.set_title('Correntes MPPT DC (A)'); ax.set_xlabel('Hora'); ax.legend(fontsize=8)

ax = axes[2]
ax.fill_between(hourly['HOUR'], hourly[TARGET], alpha=0.3, color=ACCENT)
ax.plot(hourly['HOUR'], hourly[TARGET], color=ACCENT, linewidth=2.5, marker='o', markersize=4)
ax.set_title('Potência(W) — Mediana'); ax.set_xlabel('Hora')

plt.tight_layout()
plt.show()

In [ ]:
# Correntes por string
str_hourly = df_day.groupby('HOUR')[STR_I].median().reset_index()
pal10 = sns.color_palette('tab10', len(STR_I))

fig, ax = plt.subplots(figsize=(FIG_W * 0.8, FIG_H), dpi=DPI)
for col, c in zip(STR_I, pal10):
    ax.plot(str_hourly['HOUR'], str_hourly[col], label=col, color=c, linewidth=1.5, marker='o', markersize=3)
ax.set_title('Correntes por String (Istr) — Perfil Diário Mediano (A)')
ax.set_xlabel('Hora'); ax.set_ylabel('Corrente (A)')
ax.legend(ncol=5, fontsize=8, loc='upper center', bbox_to_anchor=(0.5, -0.18))
plt.tight_layout()
plt.show()

In [ ]:
# Tensões AC (rede) por hora
ac_hourly = df_day.groupby('HOUR')[AC_V + AC_F].median().reset_index()
pal3 = sns.color_palette(PALETTE, 3)

fig, axes = plt.subplots(1, 2, figsize=(11, FIG_H), dpi=DPI)
fig.suptitle('Rede AC — Tensões e Frequências por Hora')

ax = axes[0]
for col, c in zip(AC_V, pal3):
    ax.plot(ac_hourly['HOUR'], ac_hourly[col], label=col, color=c, linewidth=1.8, marker='o', markersize=3)
ax.set_title('Tensões AC (V)'); ax.set_xlabel('Hora'); ax.legend()

ax = axes[1]
for col, c in zip(AC_F, pal3):
    ax.plot(ac_hourly['HOUR'], ac_hourly[col], label=col, color=c, linewidth=1.8, marker='o', markersize=3)
ax.set_title('Frequências AC (Hz)'); ax.set_xlabel('Hora'); ax.legend()

plt.tight_layout()
plt.show()

---
## 12. Evidência para Regressão de Potência — Corrente MPPT vs Potência

In [ ]:
# Melhor feature: I MPPT 1(A)  — ajuste potência log-log
best_feat = 'I MPPT 1(A)'
x   = df_day[best_feat].values.astype(float)
y_v = df_day[TARGET].values.astype(float)

mask_pos = (x > 0) & (y_v > 0)
log_x = np.log(x[mask_pos])
log_y = np.log(y_v[mask_pos])
slope, intercept, r_val, _, _ = stats.linregress(log_x, log_y)
alpha_hat = np.exp(intercept)

print(f'Modelo potência: {TARGET} = {alpha_hat:.4f} × {best_feat}^{slope:.4f}')
print(f'R² (espaço log-log): {r_val**2:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(FIG_W, FIG_H + 1), dpi=DPI)
fig.suptitle(f'{best_feat} vs {TARGET} — Escala Linear e Log-Log')

ax = axes[0]
ax.scatter(x[mask_pos], y_v[mask_pos], alpha=0.1, s=5, color=PRIMARY, linewidths=0)
x_line = np.linspace(x[mask_pos].min(), x[mask_pos].max(), 300)
ax.plot(x_line, alpha_hat * x_line**slope, color=ACCENT, linewidth=2,
        label=f'α={alpha_hat:.2f}, β={slope:.3f}')
ax.set_xlabel(best_feat); ax.set_ylabel(TARGET)
ax.set_title('Escala Linear'); ax.legend()

ax = axes[1]
ax.scatter(log_x, log_y, alpha=0.1, s=5, color=PRIMARY, linewidths=0)
x_ll = np.linspace(log_x.min(), log_x.max(), 200)
ax.plot(x_ll, intercept + slope * x_ll, color=ACCENT, linewidth=2,
        label=f'R²={r_val**2:.4f}')
ax.set_xlabel(f'ln({best_feat})'); ax.set_ylabel(f'ln({TARGET})')
ax.set_title('Escala Log-Log (linearização)'); ax.legend()

plt.tight_layout()
plt.show()

---
## 13. Resumo Analítico

In [ ]:
top5 = score_df.head(5)

print('=' * 65)
print('RESUMO DA ANÁLISE EXPLORATÓRIA — painels.xlsx')
print('=' * 65)
print(f"""
Dataset modo Normal (Potência>0): {len(df_day):,} registros
Período: {df_raw.hora.min().date()} → {df_raw.hora.max().date()}
Modos: Normal {(df_raw['Modo de operação']=='Normal').sum():,} | """
      f"Parado {(df_raw['Modo de operação']=='Parado').sum():,} | "
      f"Fault {(df_raw['Modo de operação']=='Fault').sum():,}"
)
print()
print('Top 5 features por Pearson |r| com Potência(W):')
for feat, row in top5.iterrows():
    print(f'  {feat:<20} Pearson={row["Pearson |r|"]:.3f}  Spearman={row["Spearman |ρ|"]:.3f}')
print(f"""
Modelo de potência log-log ({best_feat}):
  Potência ≈ {alpha_hat:.3f} × {best_feat}^{slope:.4f}   R²={r_val**2:.4f}

Observações:
  • Correntes de entrada DC (MPPT e strings) dominam a correlação (r>0.96).
  • Correntes CA de saída são igualmente correlacionadas (redundância esperada).
  • Temperatura do inversor tem correlação moderada (r≈0.62) — efeito de
    aquecimento acompanha a geração mas introduz perdas em altas temperaturas.
  • Tensões MPPT e frequências AC têm correlação baixa (<0.15).
  • Istr9 apresenta variância zero — deve ser excluída do modelo.
  • RSSI(%) (sinal de comunicação) sem relevância para a regressão.
""")